# 🧠 Amdox AI-Powered Task Optimizer
### Employee Mood & Stress Detection using Machine Learning

---

**Dataset:** [Human Stress Detection in and through Sleep](https://www.kaggle.com/datasets/laavanya/human-stress-detection-in-and-through-sleep)  
**Platform:** Kaggle  
**Goal:** Predict employee stress levels and recommend tasks accordingly

---

## 📌 Project Workflow
1. Install & Import Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Feature Engineering
6. Model Training (Multiple ML Models)
7. Model Evaluation & Comparison
8. Task Recommendation System
9. HR Alert System
10. Save & Export Model

## 📥 How to Get the Dataset

**Option 1 – Kaggle (Recommended):**
1. Go to: https://www.kaggle.com/datasets/laavanya/human-stress-detection-in-and-through-sleep
2. Download `SaYoPillow.csv`
3. Place it in the same folder as this notebook

**Option 2 – Auto-download via code** (run the cell below if you have kaggle API set up)

In [ ]:
# ============================================================
# STEP 0: Install required libraries (run once)
# ============================================================
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
# ============================================================
# STEP 1: Import All Libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Save model
import joblib

print("✅ All libraries imported successfully!")

In [ ]:
# ============================================================
# STEP 2: Load the Dataset
# ============================================================
# Dataset: SaYoPillow - Human Stress Detection
# Columns: snoring rate, respiration rate, body temperature,
#          limb movement, blood oxygen, eye movement,
#          sleeping hours, heart rate, stress level (0-4)

df = pd.read_csv('SaYoPillow.csv')

print("📊 Dataset Shape:", df.shape)
print("\n📋 First 5 rows:")
df.head()

In [ ]:
# ============================================================
# STEP 3: Exploratory Data Analysis (EDA)
# ============================================================

print("=" * 50)
print("📌 Dataset Info")
print("=" * 50)
print(df.info())

print("\n=" * 50)
print("📌 Basic Statistics")
print("=" * 50)
df.describe()

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
print(df.isnull().sum())
print("\n✅ No missing values!" if df.isnull().sum().sum() == 0 else "⚠️ Missing values found!")

In [ ]:
# ---- Rename columns for clarity ----
df.columns = [
    'snoring_rate', 'respiration_rate', 'body_temp',
    'limb_movement', 'blood_oxygen', 'eye_movement',
    'sleeping_hours', 'heart_rate', 'stress_level'
]

# ---- Stress Level Distribution ----
plt.figure(figsize=(8, 5))
stress_counts = df['stress_level'].value_counts().sort_index()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
bars = plt.bar(stress_counts.index, stress_counts.values, color=colors, edgecolor='white', linewidth=1.5)

# Add labels on bars
for bar, val in zip(bars, stress_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', fontweight='bold', fontsize=11)

plt.title('📊 Stress Level Distribution in Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Stress Level (0=None → 4=High)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0,1,2,3,4], ['0\nNone','1\nLow','2\nMedium','3\nHigh','4\nVery High'])
plt.tight_layout()
plt.savefig('stress_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nStress Level Counts:\n", stress_counts)

In [ ]:
# ---- Correlation Heatmap ----
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn_r', linewidths=0.5,
    cbar_kws={'label': 'Correlation'}
)
plt.title('🔗 Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Feature vs Stress Level Boxplots ----
features = ['heart_rate', 'blood_oxygen', 'sleeping_hours', 'snoring_rate']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
palette = ['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c']

for i, feat in enumerate(features):
    sns.boxplot(
        x='stress_level', y=feat, data=df,
        palette=palette, ax=axes[i]
    )
    axes[i].set_title(f'{feat.replace("_"," ").title()} vs Stress Level', fontweight='bold')
    axes[i].set_xlabel('Stress Level')

plt.suptitle('📈 Key Features vs Stress Level', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 4: Data Preprocessing
# ============================================================

# Separate features (X) and target (y)
X = df.drop('stress_level', axis=1)
y = df['stress_level']

print("✅ Features (X) shape:", X.shape)
print("✅ Target (y) shape:", y.shape)
print("\n📌 Feature columns:", list(X.columns))

In [ ]:
# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"🔀 Train set: {X_train.shape[0]} samples")
print(f"🔀 Test set:  {X_test.shape[0]} samples")

In [ ]:
# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")
print("   Mean of scaled train data (should be ~0):", X_train_scaled.mean().round(4))

In [ ]:
# ============================================================
# STEP 5: Feature Importance (using Random Forest)
# ============================================================

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X_train_scaled, y_train)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_temp.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
bars = plt.barh(
    importance_df['Feature'],
    importance_df['Importance'],
    color=sns.color_palette('Blues_d', len(importance_df))
)
plt.title('⭐ Feature Importance for Stress Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')

for bar, val in zip(bars, importance_df['Importance']):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔑 Top 3 most important features:")
print(importance_df.tail(3)[['Feature','Importance']].to_string(index=False))

In [ ]:
# ============================================================
# STEP 6: Train Multiple ML Models
# ============================================================

models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors':    KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':          DecisionTreeClassifier(random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', random_state=42),
}

results = {}

print("🚀 Training Models...\n")
print(f"{'Model':<28} {'Train Acc':>10} {'Test Acc':>10} {'CV Score':>10}")
print("-" * 62)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc  = accuracy_score(y_test,  model.predict(X_test_scaled))
    cv_score  = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
    results[name] = {'train_acc': train_acc, 'test_acc': test_acc, 'cv_score': cv_score}
    print(f"{name:<28} {train_acc:>10.4f} {test_acc:>10.4f} {cv_score:>10.4f}")

print("\n✅ All models trained!")

In [ ]:
# ============================================================
# STEP 7: Model Evaluation & Comparison
# ============================================================

# --- Bar chart comparison ---
results_df = pd.DataFrame(results).T

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
w = 0.25

ax.bar(x - w, results_df['train_acc'], w, label='Train Accuracy', color='#3498db', alpha=0.85)
ax.bar(x,     results_df['test_acc'],  w, label='Test Accuracy',  color='#2ecc71', alpha=0.85)
ax.bar(x + w, results_df['cv_score'],  w, label='CV Score (5-fold)', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=20, ha='right', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('🏆 Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.4, label='90% line')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Pick the Best Model ----
best_model_name = results_df['test_acc'].idxmax()
best_model      = models[best_model_name]
best_acc        = results_df.loc[best_model_name, 'test_acc']

print(f"🥇 Best Model: {best_model_name}")
print(f"   Test Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

In [ ]:
# ---- Detailed Classification Report ----
y_pred = best_model.predict(X_test_scaled)

stress_labels = ['None (0)', 'Low (1)', 'Medium (2)', 'High (3)', 'Very High (4)']

print(f"📋 Classification Report — {best_model_name}")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=stress_labels))

In [ ]:
# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1,2,3,4])
disp.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title(f'🎯 Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Stress Level')
plt.ylabel('Actual Stress Level')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 8: Task Recommendation System
# ============================================================

# Stress Level → Recommended Tasks
TASK_RECOMMENDATIONS = {
    0: {
        'mood': '😊 Excellent / No Stress',
        'tasks': [
            '✅ Lead complex projects or new initiatives',
            '✅ Creative brainstorming and strategy sessions',
            '✅ Mentoring junior team members',
            '✅ Presentations and client meetings',
        ],
        'alert': None
    },
    1: {
        'mood': '🙂 Good / Low Stress',
        'tasks': [
            '✅ Regular project work and feature development',
            '✅ Team collaboration and code reviews',
            '✅ Documentation and knowledge sharing',
            '✅ Learning new tools or upskilling',
        ],
        'alert': None
    },
    2: {
        'mood': '😐 Moderate / Medium Stress',
        'tasks': [
            '⚡ Routine maintenance or repetitive tasks',
            '⚡ Bug fixes with clear steps',
            '⚡ Attending training or webinars',
            '⚡ Take short breaks every 45 minutes',
        ],
        'alert': '🔔 Suggest: Short break / Breathing exercise'
    },
    3: {
        'mood': '😟 Poor / High Stress',
        'tasks': [
            '⚠️  Light admin tasks only',
            '⚠️  Avoid high-pressure deadlines today',
            '⚠️  Team support tasks',
            '⚠️  HR check-in recommended',
        ],
        'alert': '🚨 HR ALERT: Employee showing high stress. Consider workload review.'
    },
    4: {
        'mood': '😰 Critical / Very High Stress',
        'tasks': [
            '🛑 Do NOT assign high-priority tasks',
            '🛑 Mandatory wellness break recommended',
            '🛑 Consider reassigning current workload',
        ],
        'alert': '🚨🚨 URGENT HR ALERT: Possible burnout. Immediate counseling session required!'
    }
}

def recommend_tasks(stress_level):
    """Given a stress level (0-4), print recommendations."""
    info = TASK_RECOMMENDATIONS[stress_level]
    print("\n" + "=" * 55)
    print(f"  Employee Mood: {info['mood']}")
    print("=" * 55)
    print("  📌 Recommended Tasks:")
    for task in info['tasks']:
        print(f"     {task}")
    if info['alert']:
        print(f"\n  {info['alert']}")
    print("=" * 55)

# Demo: Try for each stress level
for level in range(5):
    recommend_tasks(level)

In [ ]:
# ============================================================
# STEP 9: HR Alert System (Live Prediction Demo)
# ============================================================
# Simulate predicting stress for new employee data

def predict_employee_stress(employee_data: dict, employee_name: str = "Employee"):
    """
    Predict stress level for a single employee and show recommendations.
    
    Parameters:
        employee_data: dict with keys matching feature names
        employee_name: name/ID of employee
    """
    features = pd.DataFrame([employee_data])[X.columns]
    scaled   = scaler.transform(features)
    stress   = best_model.predict(scaled)[0]
    
    print(f"\n👤 Employee: {employee_name}")
    print(f"   Input Data: {employee_data}")
    print(f"   🧠 Predicted Stress Level: {stress}")
    recommend_tasks(stress)
    return stress


# --- Example Employees ---

# Happy, well-rested employee
emp1 = {
    'snoring_rate': 45, 'respiration_rate': 15, 'body_temp': 98.0,
    'limb_movement': 5, 'blood_oxygen': 97, 'eye_movement': 90,
    'sleeping_hours': 8, 'heart_rate': 68
}

# Stressed employee with poor sleep
emp2 = {
    'snoring_rate': 80, 'respiration_rate': 28, 'body_temp': 99.5,
    'limb_movement': 25, 'blood_oxygen': 88, 'eye_movement': 40,
    'sleeping_hours': 3, 'heart_rate': 95
}

predict_employee_stress(emp1, "Alice (Well-rested)")
predict_employee_stress(emp2, "Bob (Poor sleep / High stress)")

In [ ]:
# ============================================================
# STEP 9b: Team Mood Analytics Dashboard
# ============================================================

# Simulate a team of 10 employees
np.random.seed(42)
team_predictions = best_model.predict(X_test_scaled[:10])
team_names = [f'Employee_{i+1}' for i in range(10)]

team_df = pd.DataFrame({
    'Name': team_names,
    'Stress_Level': team_predictions,
    'Mood': [TASK_RECOMMENDATIONS[s]['mood'] for s in team_predictions]
})

print("👥 Team Mood Analytics Dashboard")
print("=" * 50)
print(team_df.to_string(index=False))
print("\n📊 Team Summary:")
print(team_df['Stress_Level'].value_counts().sort_index())

# Pie chart of team stress
fig, ax = plt.subplots(figsize=(8, 6))
level_counts = team_df['Stress_Level'].value_counts().sort_index()
labels = [f"Level {k}: {TASK_RECOMMENDATIONS[k]['mood']}" for k in level_counts.index]
colors_pie = ['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c']

wedges, texts, autotexts = ax.pie(
    level_counts.values, labels=None,
    autopct='%1.0f%%', colors=[colors_pie[i] for i in level_counts.index],
    startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax.legend(wedges, labels, loc='lower center',
          bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9)
ax.set_title('👥 Team Mood Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('team_mood_analytics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STEP 10: Save Model & Scaler
# ============================================================

joblib.dump(best_model, 'amdox_stress_model.pkl')
joblib.dump(scaler,     'amdox_scaler.pkl')

print("✅ Model saved as: amdox_stress_model.pkl")
print("✅ Scaler saved as: amdox_scaler.pkl")
print("\n📌 To load and use later:")
print("""
import joblib
model  = joblib.load('amdox_stress_model.pkl')
scaler = joblib.load('amdox_scaler.pkl')

# Predict on new data
X_new_scaled = scaler.transform(X_new)
prediction   = model.predict(X_new_scaled)
""")

## 🎉 Project Complete!

---

### 📝 Summary

| Step | What We Did |
|------|-------------|
| 1 | Imported all necessary libraries |
| 2 | Loaded the SaYoPillow stress detection dataset |
| 3 | Performed EDA (distribution, correlations, boxplots) |
| 4 | Preprocessed data (split + scaled) |
| 5 | Identified most important features |
| 6 | Trained 6 ML models |
| 7 | Evaluated and selected the best model |
| 8 | Built a Task Recommendation System |
| 9 | Built an HR Alert System with live predictions |
| 10 | Saved model for deployment |

---

### 🚀 Next Steps (Optional Extensions)
- Connect to live wearable sensor data
- Add NLP sentiment analysis from employee text/emails
- Build a Flask/Streamlit web app for HR dashboard
- Integrate real-time email notifications for HR alerts
- Add facial expression detection (OpenCV + DeepFace)

---
*Project by Amdox | support@amdox.in*